In [3]:
import os
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [9]:
def process_all_pdf(pdf_directory):
    all_documenents = [] 
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        loader = PyMuPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            print(f" - Loaded document with {len(doc.page_content)} characters")
        all_documenents.extend(documents)

    print(f"Total chunks created from all PDFs: {len(all_documenents)}")
    return all_documenents

In [10]:
all_pdf_documents = process_all_pdf("../data")

Found 23 PDF files in ../data
Processing file: 1736941384_guidancedocument_amrdiagnostics_revised.pdf
 - Loaded document with 201 characters
 - Loaded document with 296 characters
 - Loaded document with 1275 characters
 - Loaded document with 1680 characters
 - Loaded document with 34 characters
 - Loaded document with 34 characters
 - Loaded document with 1586 characters
 - Loaded document with 1179 characters
 - Loaded document with 940 characters
 - Loaded document with 593 characters
 - Loaded document with 61 characters
 - Loaded document with 2590 characters
 - Loaded document with 892 characters
 - Loaded document with 2879 characters
 - Loaded document with 1606 characters
 - Loaded document with 953 characters
 - Loaded document with 101 characters
 - Loaded document with 2341 characters
 - Loaded document with 2808 characters
 - Loaded document with 1180 characters
 - Loaded document with 201 characters
 - Loaded document with 1870 characters
 - Loaded document with 2710 cha

In [11]:
all_pdf_documents


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\1736941384_guidancedocument_amrdiagnostics_revised.pdf', 'file_path': '..\\data\\1736941384_guidancedocument_amrdiagnostics_revised.pdf', 'total_pages': 59, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='GUIDANCE DOCUMENT \nVALIDATION OF RAPID DIAGNOSTICS \nFOR PATHOGEN IDENTIFICATION \nAND ANTIMICROBIAL SUSCEPTIBILITY TESTING (AST)\n2025\nDivision of Descriptive Research,\nIndian Council of Medical Research'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\1736941384_guidancedocument_amrdiagnostics_revised.pdf', 'file_path': '..\\data\\1736941384_guidancedocument_amrdiagnostics_revised.pdf', 'total_pages': 59, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '

In [12]:
from re import split


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Total chunks after splitting: {len(split_docs)}")

    if split_docs:
        print(f"Sample chunk metadata:\n{split_docs[0].metadata}")
        print(f"Sample chunk (first 500 characters):\n{split_docs[0].page_content[:500]}")
    return split_docs
        

In [13]:
chunks = split_documents(all_pdf_documents)

Total chunks after splitting: 6644
Sample chunk metadata:
{'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\1736941384_guidancedocument_amrdiagnostics_revised.pdf', 'file_path': '..\\data\\1736941384_guidancedocument_amrdiagnostics_revised.pdf', 'total_pages': 59, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
Sample chunk (first 500 characters):
GUIDANCE DOCUMENT 
VALIDATION OF RAPID DIAGNOSTICS 
FOR PATHOGEN IDENTIFICATION 
AND ANTIMICROBIAL SUSCEPTIBILITY TESTING (AST)
2025
Division of Descriptive Research,
Indian Council of Medical Research


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [20]:
from langchain_openai import OpenAIEmbeddings

In [32]:
class EmbeddingManager:
    """Handles document embeddings using Sentence Transformers"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Args:
            model_name (str): Name of the Sentence Transformer model to use.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the Sentence Transformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_sentence_embedding_dimension())
            print(f"Loaded Sentence Transformer model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates embeddings for a list of texts.

        Args:
            texts (List[str]): List of texts to embed.
        Returns:
            np.ndarray: Array of embeddings.
        """
        if not self.model:
            raise ValueError("Model is not loaded.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

embedding_manager = EmbeddingManager()


Model loaded successfully. Embedding dimension: 384
Loaded Sentence Transformer model: all-MiniLM-L6-v2


In [33]:
class VectorStore:
    """Manages a vector store using ChromaDB."""
    def __init__(self, collection_name: str = "pdf_documents", persistent_dir: str = "../data/vector_store"):
        """
        Args:
            collection_name (str): Name of the ChromaDB collection.
            persistent_dir (str): Directory to persist the ChromaDB database.
        """
        self.client = None
        self.collection = None
        self.collection_name = collection_name
        self.persistent_dir = persistent_dir
        self._initialize_store()


    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            os.makedirs(self.persistent_dir, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_dir)
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF Document Embeddings for RAG"})
            print(f"ChromaDB collection '{self.collection_name}' initialized at '{self.persistent_dir}'")
            print(f"existing collections: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adds documents and their embeddings to the vector store.

        Args:
            documents (List[Any]): List of documents to add.
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")
        try:
            ids = [str(uuid.uuid4()) for _ in documents]
            metadatas = [doc.metadata for doc in documents]
            texts = [doc.page_content for doc in documents]

            self.collection.add(
                ids=ids,
                embeddings=embeddings.tolist(),
                metadatas=metadatas,
                documents=texts
            )
            print(f"Added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vector_store = VectorStore()

ChromaDB collection 'pdf_documents' initialized at '../data/vector_store'
existing collections: 0


In [34]:
texts = [doc.page_content for doc in chunks]


In [35]:
texts

['GUIDANCE DOCUMENT \nVALIDATION OF RAPID DIAGNOSTICS \nFOR PATHOGEN IDENTIFICATION \nAND ANTIMICROBIAL SUSCEPTIBILITY TESTING (AST)\n2025\nDivision of Descriptive Research,\nIndian Council of Medical Research',
 'GUIDANCE DOCUMENT  \n VALIDATION OF RAPID DIAGNOSTICS  \nFOR PATHOGEN IDENTIFICATION  \nAND ANTIMICROBIAL SUSCEPTIBILITY TESTING (AST) \n2025 \n \n \n \n \n \n \n \n \n \n \n \n        \n                                                \n \nDivision of Descriptive Research              In-Vitro Diagnostics Division',
 'Standard Validation Protocol /ICMR \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nICMR, New Delhi  \n \nGUIDANCE DOCUMENT: VALIDATION OF RAPID DIAGNOSTICS FOR \nPATHOGEN IDENTIFICATION AND ANTIMICROBIAL SUSCEPTIBILITY \nTESTING (AST), 2025.  \n \n1st edition.  \n \n© 2025, Indian Council of Medical Research. All rights reserved  \n \nPublications of the ICMR can be obtained from ICMR-HQ,  \nDivision of Descriptive Research,  \nIndian Council of Medical Research  \n

In [38]:
def batched(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]


In [39]:
BATCH_SIZE = 5000

for doc_batch in batched(chunks, BATCH_SIZE):
    texts = [doc.page_content for doc in doc_batch]

    embeddings = embedding_manager.generate_embeddings(texts)

    vector_store.add_documents(doc_batch, embeddings)


Batches: 100%|██████████| 157/157 [01:55<00:00,  1.36it/s]


Adding 5000 documents to the vector store...
Added 5000 documents to the vector store.


Batches: 100%|██████████| 52/52 [00:36<00:00,  1.42it/s]


Adding 1644 documents to the vector store...
Added 1644 documents to the vector store.


In [61]:
from turtle import distance


class RAGRetriver:
    """Retrieves relevant documents from the vector store based on query embeddings."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager, top_k: int = 5):
        """
        Args:
            vector_store (VectorStore): The vector store instance.
            embedding_manager (EmbeddingManager): The embedding manager instance.
            top_k (int): Number of top similar documents to retrieve.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.top_k = top_k

    def retrieve(self, query: str,top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Retrieves the most relevant documents for a given query.

        Args:
            query (str): The input query string.
        Returns:
            List[Dict[str, Any]]: List of retrieved documents with metadata.
        """
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=self.top_k
            )
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []       
        retrieved_docs = []
        for doc, metadata in zip(results['documents'][0], results['metadatas'][0]):
            retrieved_docs.append({
                "document": doc,
                "metadata": metadata
            })
        for i, (doc_id, document, metadata, distance) in enumerate(zip(results['ids'][0], results['documents'][0], results['metadatas'][0], results['distances'][0])):
            print(f"\n\n Rank {i+1}: ID={doc_id}, Document={document}, Distance={distance:.4f}, Metadata={metadata}")
            similarity_score = 1 - distance
            print(f"Similarity Score: {similarity_score:.4f}")

        return retrieved_docs
rag_retriever = RAGRetriver(vector_store, embedding_manager, top_k=5)

In [62]:
rag_retriever.retrieve("explain PHARMACEUTICAL QUALITY SYSTEM")

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.20it/s]



 Rank 1: ID=e1a2fc4f-f7d6-43a2-bdd5-6445bf761c20, Document=Chapter 1     Pharmaceutical Quality System 
 
 
PE 009-17 (Part I) 
- 2 - 
25 August 2023 
 
 
1.3 
The size and complexity of the company’s activities should be taken into 
consideration when developing a new Pharmaceutical Quality System or 
modifying an existing one. The design of the system should incorporate 
appropriate risk management principles including the use of appropriate tools. 
While some aspects of the system can be company-wide and others site-specific, 
the effectiveness of the system is normally demonstrated at the site level.  
 
1.4 
A Pharmaceutical Quality System appropriate for the manufacture of medicinal 
products should ensure that:  
 
(i)  
Product realisation is achieved by designing, planning, implementing, 
maintaining and continuously improving a system that allows the consistent 
delivery of products with appropriate quality attributes;  
 
(ii) 
Product and process knowledge is managed thro

[{'document': 'Chapter 1     Pharmaceutical Quality System \n \n \nPE 009-17 (Part I) \n- 2 - \n25 August 2023 \n \n \n1.3 \nThe size and complexity of the company’s activities should be taken into \nconsideration when developing a new Pharmaceutical Quality System or \nmodifying an existing one. The design of the system should incorporate \nappropriate risk management principles including the use of appropriate tools. \nWhile some aspects of the system can be company-wide and others site-specific, \nthe effectiveness of the system is normally demonstrated at the site level.  \n \n1.4 \nA Pharmaceutical Quality System appropriate for the manufacture of medicinal \nproducts should ensure that:  \n \n(i)  \nProduct realisation is achieved by designing, planning, implementing, \nmaintaining and continuously improving a system that allows the consistent \ndelivery of products with appropriate quality attributes;  \n \n(ii) \nProduct and process knowledge is managed throughout all lifecycle

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY") 

llm = ChatGroq(groq_api_key=groq_api_key, model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, max_tokens=1024)

def simple_rag(query,retriver,llm,score_threshold=0):

    results = retriver.retrieve(query,top_k=5)
    filtered = [
        doc for doc in results
        if doc["metadata"].get("score", 0) >= score_threshold
    ]
    if not filtered:
        context = "No relevant documents found."
    else:
        context ="\n\n".join([doc["document"] for doc in results]) if results else "No relevant documents found."
        prompt = f"""Use the following context to answer the question in detailed manner in concisely and accurately.
            give answer in proper structure that gives all informations what the exact sentence says and as well as the concluded answer of what we can determing
            which will be needed to know the knowledge, the answer should feels 
            like human wrote it as well as in a robotic way.
            Context: {context} \n\n
            Question: {query}
            Answer:  
            
            \n\n"""
        response = llm.invoke(prompt.format(context=context, query=query))
    return response.content

In [101]:
answer= simple_rag("what are the SOFTWARES USED FOR CHECKING DRUG INTERACTIONS: ?",rag_retriever,llm)
print(f"\n\n Answer: {answer} ")

Batches: 100%|██████████| 1/1 [00:00<00:00, 66.39it/s]




 Rank 1: ID=303ace92-832e-4906-bbf1-af233260e1e9, Document=Int. J. Pharm. Sci. Rev. Res., 80(1), May – June 2023; Article No. 06, Pages: 29-40                                                     ISSN 0976 – 044X 
 
 
International Journal of Pharmaceutical Sciences Review and Research 
Available online at www.globalresearchonline.net  
©Copyright protected. Unauthorised republication, reproduction, distribution, dissemination and copying of this document in whole or in part is strictly prohibited. 
 
38 
SOFTWARES USED FOR CHECKING DRUG INTERACTIONS: 
¡ Lexicomp: 
Pharmacists working in community pharmacies or hospitals 
can use Lexicomp as a drug reference tool. Lexicomp's user-
friendly navigation, medication monographs, and drug 
interaction checks can help pharmacists work more 
efficiently and effectively across the board of the pharmacy. 
This tool is made to rapidly and effectively connect you to 
medication-related 
information, 
giving 
pharmacists, 
doctors, 
and 
nurses 
t